In [1]:
import os
!apt-get update -qq > /dev/null
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
!curl -fLO https://downloads.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz && tar xf spark-4.1.1-bin-hadoop3.tgz
%pip install -q findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  546M  100  546M    0     0  24.5M      0  0:00:22  0:00:22 --:--:-- 23.0M


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.1.1-bin-hadoop3"

In [3]:
import findspark
findspark.init()

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CollabApp").master("local[*]").config('spark.ui.port', '4050').getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [5]:
spark

## Kaggle Dataset Import (Colab)
Upload `kaggle.json`, download the dataset, extract curated tables, and load them with Spark.

In [6]:
%pip install -q kaggle

In [10]:
import os
from pathlib import Path

target_dir = Path("/root/.kaggle")
target_file = target_dir / "kaggle.json"
target_dir.mkdir(parents=True, exist_ok=True)

kaggle_config = '{"username":"bafbihmn","key":"8b5a552f4df25a9484f5b33c17bb96aa"}'
target_file.write_text(kaggle_config, encoding="utf-8")
os.chmod(target_file, 0o600)
os.environ["KAGGLE_CONFIG_DIR"] = str(target_dir)

print("Kaggle credentials configured from inline JSON.")

Kaggle credentials configured from inline JSON.


In [26]:
import os

DATASET = "bafbihmn/minecraft-item-complexity"
DATA_ROOT = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)

!kaggle datasets download -d $DATASET -p $DATA_ROOT --force

DATASET_ZIP = f"{DATA_ROOT}/minecraft-item-complexity.zip"
assert os.path.exists(DATASET_ZIP) and os.path.getsize(DATASET_ZIP) > 0, "Dataset zip not downloaded"

print(f"Downloaded: {DATASET_ZIP}")

Dataset URL: https://www.kaggle.com/datasets/bafbihmn/minecraft-item-complexity
License(s): CC0-1.0
  0% 0.00/174k [00:00<?, ?B/s]
100% 174k/174k [00:00<00:00, 286MB/s]
Downloaded: /content/data/minecraft-item-complexity.zip


In [25]:

!ls $DATA_ROOT/

minecraft-item-complexity-minestom	minestom_manifest.json
minecraft-item-complexity-minestom.zip	README.md
minecraft-item-complexity.zip


In [27]:
import os
from pathlib import Path
import zipfile

EXTRACT_ROOT = Path(DATA_ROOT)
EXTRACT_DIR = ""
CURATED_DIR = ""

os.makedirs(EXTRACT_ROOT, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, "r") as archive:
    members = [name for name in archive.namelist() if name and not name.endswith("/")]
    top_levels = sorted({name.split("/", 1)[0] for name in members if "/" in name})
    archive.extractall(EXTRACT_ROOT)

if len(top_levels) == 1 and (EXTRACT_ROOT / top_levels[0]).is_dir():
    extract_path = EXTRACT_ROOT / top_levels[0]
else:
    extract_path = EXTRACT_ROOT

EXTRACT_DIR = str(extract_path)
CURATED_DIR = str(extract_path / "curated")
curated_zip = f"{EXTRACT_DIR}/curated.zip"
if os.path.isdir(CURATED_DIR):
    pass
elif os.path.exists(curated_zip):
    os.makedirs(CURATED_DIR, exist_ok=True)
    with zipfile.ZipFile(curated_zip, "r") as archive:
        archive.extractall(CURATED_DIR)
else:
    root_tables = [path for path in Path(EXTRACT_DIR).iterdir() if path.is_dir() and not path.name.startswith(".")]
    if any((path / "_SUCCESS").exists() for path in root_tables):
        CURATED_DIR = EXTRACT_DIR
    else:
        nested_curated = next((path for path in Path(EXTRACT_DIR).rglob("curated") if path.is_dir()), None)
        if nested_curated is not None:
            CURATED_DIR = str(nested_curated)

assert os.path.isdir(CURATED_DIR), "Curated directory not found after extraction"
print("Extracted files:", sorted(os.listdir(EXTRACT_DIR)))

Extracted files: ['bridge_item_block', 'dim_biome', 'dim_block', 'dim_dimension', 'dim_entity', 'dim_item', 'fact_biome_feature', 'fact_biome_spawner', 'fact_loot_condition', 'fact_loot_entry', 'fact_loot_table', 'fact_ore_opportunity', 'meta_manifest', 'tag_members_raw']


In [28]:
from pathlib import Path

table_dirs = sorted(
    [path for path in Path(CURATED_DIR).iterdir() if path.is_dir() and not path.name.startswith(".")],
    key=lambda path: path.name
)
table_names = [path.name for path in table_dirs]
assert table_names, "No curated parquet tables found"

tables = {name: spark.read.parquet(str(Path(CURATED_DIR) / name)) for name in table_names}

for sample in ["meta_manifest", "dim_item", "fact_loot_entry"]:
    if sample in tables:
        print(sample, tables[sample].count())

tables["dim_item"].limit(10)

meta_manifest 1
dim_item 1505
fact_loot_entry 2156


corresponding_block_id,is_enchantable,is_fire_resistant,item_id,max_damage,max_stack_size,numeric_id,translation_key
NULL,NULL,NULL,minecraft:air,NULL,NULL,0,block.minecraft.air
minecraft:stone,NULL,NULL,minecraft:stone,NULL,NULL,1,block.minecraft.s...
minecraft:granite,NULL,NULL,minecraft:granite,NULL,NULL,2,block.minecraft.g...
minecraft:polishe...,NULL,NULL,minecraft:polishe...,NULL,NULL,3,block.minecraft.p...
minecraft:diorite,NULL,NULL,minecraft:diorite,NULL,NULL,4,block.minecraft.d...
minecraft:polishe...,NULL,NULL,minecraft:polishe...,NULL,NULL,5,block.minecraft.p...
minecraft:andesite,NULL,NULL,minecraft:andesite,NULL,NULL,6,block.minecraft.a...
minecraft:polishe...,NULL,NULL,minecraft:polishe...,NULL,NULL,7,block.minecraft.p...
minecraft:deepslate,NULL,NULL,minecraft:deepslate,NULL,NULL,8,block.minecraft.d...
minecraft:cobbled...,NULL,NULL,minecraft:cobbled...,NULL,NULL,9,block.minecraft.c...


In [ ]:
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main" \
  | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt update \
  && sudo apt install ngrok

deb https://ngrok-agent.s3.amazonaws.com bookworm main
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:5 https://cli.github.com/packages stable InRelease                         
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease     
Hit:8 https://ngrok-agent.s3.amazonaws.com bookworm InRelease       
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,745 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,903 kB]
Fetched 12.7 MB i

In [ ]:
get_ipython().system_raw('ngrok config add-authtoken 38WkYE78wPcHv1mxbWKEvRKOiA3_7ksTxxPSfcsQNGzvX686n') # <- change here
get_ipython().system_raw('ngrok http http://localhost:4050 &')

In [ ]:
!curl -s http://localhost:4040/api/tunnels

{"tunnels":[{"name":"command_line","ID":"fbb7c81e7658c738cb674261b6a64ed8","uri":"/api/tunnels/command_line","public_url":"https://humourless-extrasystolic-ashlee.ngrok-free.dev","proto":"https","config":{"addr":"http://localhost:4050","inspect":true},"metrics":{"conns":{"count":0,"gauge":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0},"http":{"count":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0}}}],"uri":"/api/tunnels"}


# Project Scope: Minecraft Item Complexity Analysis

## 1. Project Overview
The objective of this project is to quantify the "Complexity Score" of Vanilla Minecraft items by analyzing the entire item dependency graph. By leveraging **Apache Spark** for distributed data processing, we will ingest heterogeneous datasets (recipes, loot tables, biome generation, and dimension data) to construct a recursive cost model.

Unlike simple raw material counting, this project calculates complexity based on **Acquisition Effort**. This metric aggregates the recursive cost of crafting components, the rarity of loot drops, the specific biome availability, and the dimensional gating (Overworld vs. Nether vs. End) required to obtain an item.

## 2. Data Ingestion & Sources
The project will require the extraction and loading of semi-structured data (JSON/CSV) into the Hadoop File System (HDFS) or local file system for Spark processing.
*   **Recipes Dataset:** Standard crafting table, furnace, and smithing table recipes (Input/Output mapping).
*   **Loot Tables:** Probabilistic data defining drop rates from mobs and structural chests.
*   **World Generation Data:**
    *   **Biome Distribution:** Frequency of biomes where specific blocks/mobs are found.
    *   **Ore Distribution:** Y-level availability and vein size.
*   **Dimensional Metadata:** Categorization of items based on required dimensions (Overworld, Nether, The End).

## 3. Core Transformations & Analysis
The core logic relies on transforming relational data into a Directed Acyclic Graph (DAG) of dependencies to perform iterative calculations.

### 3.1. Data Cleaning & Normalization
*   Flattening nested JSON structures (e.g., complex loot tables) into Spark DataFrames.
*   Standardizing item IDs (e.g., `minecraft:log` vs `minecraft:oak_log`) to ensure join integrity.
*   Handling metadata variations (durability, potion effects) by grouping them into base item types.

### 3.2. Complexity Calculation Algorithms
We will implement a scoring algorithm using Spark (DataFrames or GraphX) that considers the following vectors:

1.  **Recursive Crafting Cost:**
    *   Iteratively resolving dependencies (e.g., An *Enchanting Table* inherits the cost of *Diamonds*, *Obsidian*, and *Books*).
    *   *Books* inherit the cost of *Leather* and *Paper* (Sugar Cane).
2.  **Loot Rarity Weighting:**
    *   Items obtainable only via RNG (Random Number Generation) drops are penalized based on probability (e.g., `1 / Drop_Chance`).
3.  **Environmental Scarcity:**
    *   **Biome Weight:** Items restricted to rare biomes (e.g., Mushroom Islands, Deep Dark) receive a higher complexity multiplier.
    *   **Dimensional Gate:** Items requiring access to the Nether or The End apply a distinct multiplier to represent the "risk" and progression requirements.
4.  **Tooling Requirements:**
    *   Adjusting scores based on required harvest levels (e.g., Obsidian requires a Diamond Pickaxe, adding the Diamond Pickaxe's complexity to the Obsidian's acquisition cost).

## 4. Visualization & Reporting
The final output will be a set of analytical visualizations generated from the processed Spark results:
*   **Complexity Heatmap:** A visual distribution of items from "Trivial" (Dirt) to "Legendary" (Nether Star/Beacon).
*   **Dependency Trees:** Force-directed graph visualizations for top-tier items (e.g., The Beacon) showing the breakdown of their resource chain.
*   **Category Analysis:** Comparative bar charts showing average complexity per item category (Food vs. Weapons vs. Redstone).
*   **"Most Expensive" List:** A ranked table of the top 20 most complex items in the game based on the calculated algorithm.

## 5. Technical Stack
*   **Storage:** HDFS (or local file system for development).
*   **Processing Engine:** Apache Spark (PySpark or Scala).
*   **Libraries:** Spark SQL (for structured data), Spark GraphX (optional, for dependency trees).
*   **Visualization:** Matplotlib/Seaborn (Python) or export to Tableau/PowerBI.

## 6. Out of Scope
To ensure project feasibility within the timeframe, the following are excluded:
*   **Modded Content:** Analysis is restricted to Vanilla Minecraft (Java Edition).
*   **Dynamic Difficulty:** We will not account for player skill, speedrunning strategies, or trading hall optimizations (Villager trading will be treated as a static cost or excluded if data is unavailable).
*   **Durability/Fuel Consumption:** We will assume 1 unit of fuel is used per furnace operation, without calculating the infinite complexity of fuel farming (e.g., bamboo farms).
*   **Circular Dependencies:** Logic will be implemented to break infinite loops (e.g., Bucket fills Water -> Water Bucket empties -> Bucket) by assigning base values to reusable tools.